<a href="https://colab.research.google.com/github/anjicx/ComorbidityNetworkGNN/blob/main/TemporalGNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch-geometric

import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    auc,
)
from google.colab import drive
drive.mount('/content/drive')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
#LOADIG DATA


# adjacency matrices
adj_matrices = {}
for i in range(1, 9):
    path = f'/content/drive/MyDrive/Adj_Matrix_Age_ICD_Male/Adj_Matrix_Male_ICD_age_{i}.csv'
    adj_matrices[i] = pd.read_csv(path, sep=r"\s+", header=None, engine="python")

# ICD codes
icd_df = pd.read_csv('/content/drive/MyDrive/AdjacencyMatrices/ICD10_Diagnoses_All.csv')
icd_codes = icd_df['icd_code'].tolist()

# prevalence
prev_df = pd.read_csv('/content/drive/MyDrive/AdjacencyMatrices/Prevalence_Sex_Age_Year_ICD.csv')

male_prev = (
    prev_df[prev_df['sex'] == 'Male']
    .groupby(['Age_Group', 'icd_code'], as_index=False)['p']
    .mean()
)

age_labels = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79"]

prev_features = {}
for idx, age in enumerate(age_labels, 1):
    age_data = male_prev[male_prev['Age_Group'] == age].set_index('icd_code')['p']
    prev_vector = np.array([age_data.get(code, 0.0) for code in icd_codes], dtype=float)
    prev_features[idx] = prev_vector
    print(f"Age group {idx} ({age}): {(prev_vector > 0).sum()} diseases have prevalence")

# ICD block one-hot
block_map = {
    'A': 0, 'B': 0,
    'C': 1, 'D': 1,
    'E': 2,
    'F': 3,
    'G': 4,
    'H': 5,
    'I': 6,
    'J': 7,
    'K': 8,
    'L': 9,
    'M': 10,
    'N': 11,
    'O': 12,
    'P': 13,
    'Q': 14,
    'R': 15,
    'S': 16, 'T': 16,
    'V': 17, 'W': 17, 'X': 17, 'Y': 17,
    'Z': 18
}

num_blocks = max(block_map.values()) + 1
blocks = np.zeros((len(icd_codes), num_blocks), dtype=float)

for j, code in enumerate(icd_codes):
    first = code[0].upper()
    if first in block_map:
        blocks[j, block_map[first]] = 1.0



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.8 MB/s eta 0:00:00
Mounted at /content/drive
Device: cuda
Age group 1 (0-9): 883 diseases have prevalence
Age group 2 (10-19): 897 diseases have prevalence
Age group 3 (20-29): 929 diseases have prevalence
Age group 4 (30-39): 932 diseases have prevalence
Age group 5 (40-49): 953 diseases have prevalence
Age group 6 (50-59): 935 diseases have prevalence
Age group 7 (60-69): 925 diseases have prevalence
Age group 8 (70-79): 883 diseases have prevalence


THIS IS IF USING SAMPLED TRAINING DATASET!!!!!!!!!!!!

In [ ]:

NUM_AGES = 8
NUM_NODES = len(icd_codes)
#BUILDING FULL GRAPHS
def build_full_graphs(adj_matrices, prev_features, blocks, add_degree_feature=True):
    graphs = {}
    binary_adj = {}

    for age in range(1, NUM_AGES + 1):
        raw = adj_matrices[age].values.astype(float)

        # 1/0 adj matrix- we dont need the full one->THIS CAN CHANGE IF U DECIDE TO USE IMPORT GRAPH STRUCTURE AND USE PRECOMPUTED CENTR MEASURES
        adj_bin = (raw > 0).astype(np.int64)
        np.fill_diagonal(adj_bin, 0)
        adj_bin = np.maximum(adj_bin, adj_bin.T)
        binary_adj[age] = adj_bin

        # edge_index for RandomSplit
        rows, cols = np.where(
            (adj_bin > 0) &
            np.triu(np.ones_like(adj_bin, dtype=bool), k=1)
        )

        if len(rows) == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)
        else:
            edge_index = torch.tensor(
                np.vstack([
                    np.concatenate([rows, cols]),
                    np.concatenate([cols, rows])
                ]),
                dtype=torch.long
            )

        prev_vec = prev_features[age].reshape(-1, 1)  # [N,1]
        x_parts = [prev_vec, blocks]

        if add_degree_feature:
            degree = adj_bin.sum(axis=1).reshape(-1, 1).astype(float)
            x_parts.append(degree)

        x = np.hstack(x_parts)

        graphs[age] = Data(
            x=torch.tensor(x, dtype=torch.float),
            edge_index=edge_index,
            num_nodes=NUM_NODES
        )

        print(
            f"Age {age}: x shape={graphs[age].x.shape}, "
            f"edges={edge_index.shape[1] // 2}"
        )

    return graphs, binary_adj


graphs, binary_adj = build_full_graphs(
    adj_matrices=adj_matrices,
    prev_features=prev_features,
    blocks=blocks,
    add_degree_feature=True
)

BASE_IN_CHANNELS = graphs[1].x.shape[1]
print("Base feature dim:", BASE_IN_CHANNELS)



Age 1: x shape=torch.Size([1080, 21]), edges=425
Age 2: x shape=torch.Size([1080, 21]), edges=120
Age 3: x shape=torch.Size([1080, 21]), edges=312
Age 4: x shape=torch.Size([1080, 21]), edges=631
Age 5: x shape=torch.Size([1080, 21]), edges=1489
Age 6: x shape=torch.Size([1080, 21]), edges=2456
Age 7: x shape=torch.Size([1080, 21]), edges=3224
Age 8: x shape=torch.Size([1080, 21]), edges=3663
Base feature dim: 21


In [ ]:

#G1..G6/G7/G8
def build_prefix_tasks(graphs):
    tasks = []
    for target_age in range(2, NUM_AGES + 1):
        history_graphs = [graphs[a] for a in range(1, target_age)]
        tasks.append({
            "target_age": target_age,
            "history": history_graphs,
            "target_graph": graphs[target_age]
        })
    return tasks


tasks = build_prefix_tasks(graphs)
for t in tasks:
    hist_ages = list(range(1, t["target_age"]))
    print(f"Predict G{t['target_age']} from history {hist_ages}")

train_tasks = [t for t in tasks if t["target_age"] <= 6]
val_task = [t for t in tasks if t["target_age"] == 7][0]
test_task = [t for t in tasks if t["target_age"] == 8][0]

print("Train target ages:", [t["target_age"] for t in train_tasks])
print("Val target age:", val_task["target_age"])
print("Test target age:", test_task["target_age"])

# SAMPLING NEGATIVES FOR TRAINING PART !!!!!!!
def build_sampled_train_labels(graph: Data, seed: int, neg_sampling_ratio=1.0):
    set_seed(seed)

    transform = RandomLinkSplit(
        num_val=0.0,
        num_test=0.0,
        is_undirected=True,
        add_negative_train_samples=True,
        neg_sampling_ratio=neg_sampling_ratio,
        split_labels=False
    )

    train_data, _, _ = transform(graph)
    return train_data


Predict G2 from history [1]
Predict G3 from history [1, 2]
Predict G4 from history [1, 2, 3]
Predict G5 from history [1, 2, 3, 4]
Predict G6 from history [1, 2, 3, 4, 5]
Predict G7 from history [1, 2, 3, 4, 5, 6]
Predict G8 from history [1, 2, 3, 4, 5, 6, 7]
Train target ages: [2, 3, 4, 5, 6]
Val target age: 7
Test target age: 8


In [ ]:
BASE_SEED=42

sampled_train_targets = {}
for task in train_tasks:
    age = task["target_age"]
    sampled_train_targets[age] = build_sampled_train_labels(
        graph=task["target_graph"],
        seed=BASE_SEED + age,
        neg_sampling_ratio=30.0#HERE WE SET HOW MANY NEGATIVES PER 1 POSITIVE!!!!!!!!!!!!!!!
    )
    d = sampled_train_targets[age]
    pos = int(d.edge_label.sum().item())
    neg = int(d.edge_label.numel() - pos)
    print(f"Train target G{age}: positives={pos}, negatives={neg}, neg/pos={neg/max(pos,1):.2f}")

#NODE FEATURES FRU

class NodeFeatureGRU(nn.Module):
    def __init__(
        self,
        base_in_channels,
        num_nodes,
        id_emb_dim=16,#16 IS EMBEDDING DIMENSIONN
        gru_hidden=64,#64 NEURONS IN HIDDEN LAYER
        dropout=0.1#DROPOUT RATE
    ):
        super().__init__()
        self.num_nodes = num_nodes

        # optional learnable node identity embedding
        self.node_id_emb = nn.Embedding(num_nodes, id_emb_dim)

        self.gru = nn.GRU(
            input_size=base_in_channels + id_emb_dim,
            hidden_size=gru_hidden,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(dropout)
        self.next_node_proj = nn.Linear(gru_hidden, gru_hidden)

        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * gru_hidden, gru_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(gru_hidden, 1)
        )

    def build_history_tensor(self, history_graphs):
        """
        history_graphs: list of graphs
        returns seq of shape [N, T, F_total]
        """
        seq_list = []

        node_ids = torch.arange(self.num_nodes, device=history_graphs[0].x.device)
        id_emb = self.node_id_emb(node_ids)  # [N, id_emb_dim]

        for g in history_graphs:
            x_t = torch.cat([g.x, id_emb], dim=-1)  # [N, F+id]
            seq_list.append(x_t)

        seq = torch.stack(seq_list, dim=1)  # [N, T, F_total]
        return seq

    def predict_next_node_embeddings(self, history_graphs):
        seq = self.build_history_tensor(history_graphs)  # [N, T, F_total]
        out, _ = self.gru(seq)                           # [N, T, H]
        last_out = out[:, -1, :]                        # [N, H]
        last_out = self.dropout(last_out)
        z_next = self.next_node_proj(last_out)          # [N, H]
        return z_next

    def decode_edges(self, z_next, edge_label_index):
        src = z_next[edge_label_index[0]]               # [E, H]
        dst = z_next[edge_label_index[1]]               # [E, H]
        pair = torch.cat([src, dst], dim=-1)            # [E, 2H]
        logits = self.edge_mlp(pair).squeeze(-1)        # [E]
        return logits

    def forward(self, history_graphs, edge_label_index):
        z_next = self.predict_next_node_embeddings(history_graphs)
        logits = self.decode_edges(z_next, edge_label_index)
        return logits

Train target G2: positives=120, negatives=3600, neg/pos=30.00
Train target G3: positives=312, negatives=9360, neg/pos=30.00
Train target G4: positives=631, negatives=18930, neg/pos=30.00
Train target G5: positives=1489, negatives=44670, neg/pos=30.00
Train target G6: positives=2456, negatives=73680, neg/pos=30.00


In [ ]:


def clone_to_device_graph(graph, device):#cpu/gpu data
    g = copy.copy(graph)
    g.x = graph.x.to(device)
    g.edge_index = graph.edge_index.to(device)
    if hasattr(graph, "edge_label_index"):
        g.edge_label_index = graph.edge_label_index.to(device)
    if hasattr(graph, "edge_label"):
        g.edge_label = graph.edge_label.to(device)
    return g

def move_history_to_device(history_graphs, device):
    return [clone_to_device_graph(g, device) for g in history_graphs]

def build_all_undirected_pairs(num_nodes, device):
    rows, cols = torch.triu_indices(num_nodes, num_nodes, offset=1, device=device)
    return torch.stack([rows, cols], dim=0)

def get_full_y_true_from_adj(adj_bin):
    rows, cols = np.triu_indices(adj_bin.shape[0], k=1)
    y_true = adj_bin[rows, cols].astype(np.int64)
    return y_true

def precision_at_k(y_true, y_score, k):#K@precision
    idx = np.argsort(-y_score)[:k]
    return float(y_true[idx].mean()) if len(idx) > 0 else 0.0

def recall_at_k(y_true, y_score, k):#K@recall
    idx = np.argsort(-y_score)[:k]
    tp_at_k = y_true[idx].sum()
    total_pos = y_true.sum()
    return float(tp_at_k / total_pos) if total_pos > 0 else 0.0



In [ ]:

def score_all_pairs(model, history_graphs, batch_size=50000):#SCORING ALL UNDIRECTED FOR NEXT GRAPH
    model.eval()
    with torch.no_grad():
        z_next = model.predict_next_node_embeddings(history_graphs)  # [N, H]
        num_nodes = z_next.size(0)
        all_pairs = build_all_undirected_pairs(num_nodes, z_next.device)

        probs_all = []
        num_pairs = all_pairs.size(1)

        for start in range(0, num_pairs, batch_size):
            end = min(start + batch_size, num_pairs)
            pair_batch = all_pairs[:, start:end]
            logits = model.decode_edges(z_next, pair_batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            probs_all.append(probs)

        y_score = np.concatenate(probs_all)
        return all_pairs.cpu(), y_score


In [ ]:

def evaluate_full_matrix(model, history_graphs, target_adj_bin, batch_size=50000): #FINAL EVALUAITON PART
    pair_index_cpu, y_score = score_all_pairs(model, history_graphs, batch_size=batch_size)
    y_true = get_full_y_true_from_adj(target_adj_bin)

    roc_auc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)

    precision, recall, _ = precision_recall_curve(y_true, y_score)
    pr_auc = auc(recall, precision)

    pos = int(y_true.sum())
    total = len(y_true)
    neg = total - pos

    return {
        "num_pairs": total,
        "num_pos": pos,
        "num_neg": neg,
        "pos_rate": pos / total,
        "ap": ap,
        "pr_auc": pr_auc,
        "ap_baseline": pos / total,
        "precision_at_100": precision_at_k(y_true, y_score, 100),
        "precision_at_500": precision_at_k(y_true, y_score, 500),
        "recall_at_100": recall_at_k(y_true, y_score, 100),
        "recall_at_500": recall_at_k(y_true, y_score, 500),
        "y_true": y_true,
        "y_score": y_score,
        "pair_index_cpu": pair_index_cpu
    }


In [ ]:
import copy
import pandas as pd
import torch
import torch.nn as nn

def run_one_seed(base_seed=42, epochs=200, neg_ratio=30.0):
    # -----------------------------
    # 1) reset randomness
    # -----------------------------
    set_seed(base_seed)

    # -----------------------------
    # 2) build sampled train targets ONCE for this seed
    # -----------------------------
    sampled_train_targets = {}
    for task in train_tasks:
        age = task["target_age"]
        sampled_train_targets[age] = build_sampled_train_labels(
            graph=task["target_graph"],
            seed=base_seed + age,
            neg_sampling_ratio=neg_ratio
        )

    # -----------------------------
    # 3) reset again before model init
    # -----------------------------
    set_seed(base_seed)

    model = NodeFeatureGRU(
        base_in_channels=BASE_IN_CHANNELS,
        num_nodes=NUM_NODES,
        id_emb_dim=16,
        gru_hidden=64,
        dropout=0.1
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()

    best_val_ap = -1.0
    best_state = None

    # optional histories
    train_loss_hist = []
    val_ap_hist = []

    # -----------------------------
    # 4) train
    # -----------------------------
    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0

        for task in train_tasks:
            target_age = task["target_age"]

            history_graphs = move_history_to_device(task["history"], device)
            train_target = clone_to_device_graph(sampled_train_targets[target_age], device)

            optimizer.zero_grad()

            logits = model(history_graphs, train_target.edge_label_index)
            loss = criterion(logits, train_target.edge_label.float())

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        mean_train_loss = total_train_loss / len(train_tasks)
        train_loss_hist.append(mean_train_loss)

        # validation on G7
        val_history = move_history_to_device(val_task["history"], device)
        val_metrics = evaluate_full_matrix(
            model=model,
            history_graphs=val_history,
            target_adj_bin=binary_adj[7],
            batch_size=50000
        )

        val_ap_hist.append(val_metrics["ap"])

        if val_metrics["ap"] > best_val_ap:
            best_val_ap = val_metrics["ap"]
            best_state = copy.deepcopy(model.state_dict())

    # load best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)

    # -----------------------------
    # 5) final test on G8
    # -----------------------------
    test_history = move_history_to_device(test_task["history"], device)
    test_metrics = evaluate_full_matrix(
        model=model,
        history_graphs=test_history,
        target_adj_bin=binary_adj[8],
        batch_size=50000
    )

    return {
        "seed": base_seed,
        "best_val_ap": best_val_ap,
        "test_ap": test_metrics["ap"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_precision_at_100": test_metrics["precision_at_100"],
        "test_precision_at_500": test_metrics["precision_at_500"],
        "test_recall_at_100": test_metrics["recall_at_100"],
        "test_recall_at_500": test_metrics["recall_at_500"],
        "test_ap_baseline": test_metrics["ap_baseline"],
    }

In [ ]:
seed_list = [42, 43, 44, 45, 46]

all_results = []
for s in seed_list:
    print(f"Running seed {s}...")
    res = run_one_seed(base_seed=s, epochs=200, neg_ratio=30.0)
    all_results.append(res)

results_df = pd.DataFrame(all_results)
print("\nPer-seed results:")
print(results_df)

Running seed 42...
Running seed 43...
Running seed 44...
Running seed 45...
Running seed 46...

Per-seed results:
   seed  best_val_ap   test_ap  test_pr_auc  test_precision_at_100  \
0    42     0.527961  0.533320     0.533206                   0.86   
1    43     0.500025  0.551500     0.551405                   0.91   
2    44     0.506914  0.535161     0.535042                   0.84   
3    45     0.489216  0.538793     0.538696                   0.92   
4    46     0.534393  0.550515     0.550432                   0.92   

   test_precision_at_500  test_recall_at_100  test_recall_at_500  \
0                  0.834            0.023478            0.113841   
1                  0.818            0.024843            0.111657   
2                  0.804            0.022932            0.109746   
3                  0.812            0.025116            0.110838   
4                  0.844            0.025116            0.115206   

   test_ap_baseline  
0          0.006287  
1          0

In [ ]:
summary_df = results_df.drop(columns=["seed"]).agg(["mean", "std"]).T
summary_df["cv_percent"] = 100 * summary_df["std"] / summary_df["mean"].replace(0, float("nan"))
for metric in summary_df.index:
    mean_val = summary_df.loc[metric, "mean"]
    std_val = summary_df.loc[metric, "std"]
    print(f"{metric}: {mean_val:.6f} ± {std_val:.6f}")

best_val_ap: 0.511702 ± 0.019001
test_ap: 0.541858 ± 0.008589
test_pr_auc: 0.541756 ± 0.008601
test_precision_at_100: 0.890000 ± 0.037417
test_precision_at_500: 0.822400 ± 0.016334
test_recall_at_100: 0.024297 ± 0.001021
test_recall_at_500: 0.112258 ± 0.002230
test_ap_baseline: 0.006287 ± 0.000000


#SHUFFLING LABELS TO CHECK IF THERE IS SOME DATA LEAKAGE

In [ ]:
def shuffle_edge_labels(data, seed=42):
#AP ALMOST BASELINE AS A RESULT
    d = copy.copy(data)
    rng = torch.Generator()
    rng.manual_seed(seed)

    perm = torch.randperm(d.edge_label.numel(), generator=rng)
    d.edge_label = d.edge_label[perm]
    return d


shuffled_train_targets = {}
for task in train_tasks:
    age = task["target_age"]
    shuffled_train_targets[age] = shuffle_edge_labels(
        sampled_train_targets[age],
        seed=BASE_SEED + 1000 + age
    )

USE_SHUFFLED_LABELS = True
target_source = shuffled_train_targets if USE_SHUFFLED_LABELS else sampled_train_targets


model = NodeFeatureGRU(
    base_in_channels=BASE_IN_CHANNELS,
    num_nodes=NUM_NODES,
    id_emb_dim=16,
    gru_hidden=64,
    dropout=0.1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

train_loss_hist = []
val_ap_hist = []
val_auc_hist = []
val_prauc_hist = []

best_val_ap = -1.0
best_state = None

for epoch in range(1, 201):
    model.train()
    total_train_loss = 0.0

    for task in train_tasks:
        target_age = task["target_age"]

        history_graphs = move_history_to_device(task["history"], device)
        train_target = clone_to_device_graph(target_source[target_age], device)

        optimizer.zero_grad()

        logits = model(history_graphs, train_target.edge_label_index)
        loss = criterion(logits, train_target.edge_label.float())

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    mean_train_loss = total_train_loss / len(train_tasks)
    train_loss_hist.append(mean_train_loss)

    val_history = move_history_to_device(val_task["history"], device)
    val_metrics = evaluate_full_matrix(
        model=model,
        history_graphs=val_history,
        target_adj_bin=binary_adj[7],
        batch_size=50000
    )

    val_ap_hist.append(val_metrics["ap"])
    val_prauc_hist.append(val_metrics["pr_auc"])

    if val_metrics["ap"] > best_val_ap:
        best_val_ap = val_metrics["ap"]
        best_state = copy.deepcopy(model.state_dict())

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={mean_train_loss:.4f} | "
            f"VAL(G7 full) AP={val_metrics['ap']:.4f} | "
            f"PR-AUC={val_metrics['pr_auc']:.4f} | "
            f"AP baseline={val_metrics['ap_baseline']:.8f}"
        )

if best_state is not None:
    model.load_state_dict(best_state)

# final test on G8
test_history = move_history_to_device(test_task["history"], device)
test_metrics = evaluate_full_matrix(
    model=model,
    history_graphs=test_history,
    target_adj_bin=binary_adj[8],
    batch_size=50000
)

print("\n===== SHUFFLE-LABEL TEST RESULTS =====")
print(f"VAL best AP : {best_val_ap:.6f}")
print(f"TEST AP     : {test_metrics['ap']:.6f}")
print(f"TEST baseline AP : {test_metrics['ap_baseline']:.6f}")
print(f"TEST PR-AUC : {test_metrics['pr_auc']:.6f}")
print(f"Precision@100: {test_metrics['precision_at_100']:.6f}")
print(f"Recall@100   : {test_metrics['recall_at_100']:.6f}")